# 01 THINGS-EEG2 audit, folds and label budgets

**Kaggle:** CPU, Internet on. **Inputs:** none, or the THINGS-EEG2 dataset `cross-subject-brain-encoding-eeg` (09g).

Checks what the THINGS-EEG2 release contains, that every person saw the images in the same order, and measures the
noise ceiling; then writes the hold-one-person-out folds and the nested label budgets that later notebooks use. No
model is fitted here.

In [ ]:
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
import shutil

import numpy as np
import pandas as pd

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.utils import config as C

ATTACHED = C.EEG_RAW.is_dir() and any(C.EEG_RAW.rglob("preprocessed_eeg_test.npy"))
WORK = C.EEG_RAW if ATTACHED else Path("/kaggle/tmp/things_eeg2" if IS_KAGGLE else C.PROJECT_ROOT / "data" / "raw" / "things_eeg2")
print("EEG:", "attached at" if ATTACHED else "downloading per subject into", WORK)
meta = te.load_image_metadata(C.EEG_RAW if ATTACHED else None)
print({k: len(v) for k, v in meta.items()})

## Every subject

For each subject this loads the training and test files, checks their
shapes, channels and time axis, and keeps only what later cells need: the
test data (about 0.1 GB per subject in float32) and a few summary numbers.
The training data is checked and then released.

In [ ]:
rows, problems, tests = [], [], {}
ref_channels = ref_times = None

for s in C.EEG_SUBJECTS:
    if not ATTACHED and te.subject_file(WORK, s, "test") is None:
        te.download_subject(s, WORK)
    row = {"subject": s}
    for part in ("training", "test"):
        loaded = te.load_part(te.subject_file(WORK, s, part))
        problems += [f"{s} {p}" for p in te.check_part(loaded, part)]
        ref_channels = ref_channels or loaded["ch_names"]
        ref_times = loaded["times"] if ref_times is None else ref_times
        if loaded["ch_names"] != ref_channels:
            problems.append(f"{s} {part}: channel order differs from {C.EEG_SUBJECTS[0]}")
        if not np.allclose(loaded["times"], ref_times):
            problems.append(f"{s} {part}: time axis differs")
        row[f"{part}_shape"] = "x".join(map(str, loaded["data"].shape))
        row[f"{part}_sd"] = float(loaded["data"].std())
        if part == "test":
            tests[s] = loaded["data"]
        del loaded
    rows.append(row)
    if not ATTACHED:
        shutil.rmtree(WORK / s, ignore_errors=True)
    print(f"{s}: done")

manifest = pd.DataFrame(rows)
manifest.to_csv(C.DATA_PROCESSED / C.EEG_MANIFEST, index=False)
print("channels:", ref_channels)
print(f"time axis: {ref_times[0]:.2f} to {ref_times[-1]:.2f} s, {len(ref_times)} points")
manifest

Expected: training (16540, 4, 17, 100) and test (200, 80, 17, 100) for all
ten subjects, the same 17 channel names in the same order, and a time axis
from -0.20 to 0.79 s. Any problem below means an incomplete or wrong
download, which has to be fixed, not worked around.

In [ ]:
print("\n".join(problems) if problems else "Every subject matches the published release.")

## Same image order for everyone

The files carry no image ids; the order comes from the shared metadata
file. So the check is statistical. For each subject, the response to each
test image (repetition average, 80 to 200 ms, per channel) forms a profile.
Two subjects whose images line up should have clearly correlated profiles,
while shuffling one subject's images gives the null. Expect every pair well
above its shuffled 99th percentile. A pair near the null means one subject's
order differs, and loading by position would mix up images.

In [ ]:
profiles = {s: te.image_profile(t, ref_times, C.ORDER_CHECK_WINDOW_S) for s, t in tests.items()}
order = pd.DataFrame(te.order_agreement(profiles, n_perm=200, seed=C.SEED))
order.to_csv(C.METRICS_DIR / "01_order_agreement.csv", index=False)
bad = order[order["r"] <= order["null_p99"]]
print(f"{len(order)} pairs; median r {order['r'].median():.3f}; median shuffled 99th percentile {order['null_p99'].median():.3f}")
print("Order agrees for every pair." if bad.empty else f"Pairs at the null:\n{bad}")

## Noise ceiling

Split-half reliability over the 80 test repetitions (100 random splits,
seed 13), stepped down to one trial and back up with Spearman-Brown. For
each channel and time point the file holds the reliability and the highest
correlation a model can reach with a 4-trial and an 80-trial average. The
THINGS-EEG2 paper's encoders peak at 110 to 130 ms, so the ceiling should
peak there too, near zero before the image appears, and be highest over
occipital channels.

In [ ]:
from cross_subject_encoding.eval.noise_ceiling import noise_ceiling

per_subject = {s: noise_ceiling(t, C.NC_SPLITS, C.SEED, tuple(C.EEG_TEST_REPS_SCORED)) for s, t in tests.items()}
keys = list(next(iter(per_subject.values())))
np.savez_compressed(C.DATA_PROCESSED / C.EEG_NOISE_CEILING, subjects=np.array(C.EEG_SUBJECTS),
                    ch_names=np.array(ref_channels), times=ref_times,
                    **{k: np.stack([per_subject[s][k] for s in C.EEG_SUBJECTS]) for k in keys})
mean80 = np.nanmean(np.stack([per_subject[s]["max_r_80"] for s in C.EEG_SUBJECTS]), axis=(0, 1))
pre = ref_times < 0
print(f"max achievable r (80 trials), mean over subjects and channels: before onset {np.nanmean(mean80[pre]):.3f}, "
      f"peak {np.nanmax(mean80):.3f} at {ref_times[np.nanargmax(mean80)] * 1000:.0f} ms")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for n, ax in zip(C.EEG_TEST_REPS_SCORED, axes):
    stack = np.stack([per_subject[s][f"max_r_{n}"] for s in C.EEG_SUBJECTS])  # subjects, channels, time
    for s_i in range(stack.shape[0]):
        ax.plot(ref_times * 1000, np.nanmean(stack[s_i], 0), color="0.7", lw=0.8)
    ax.plot(ref_times * 1000, np.nanmean(stack, (0, 1)), color="k", lw=2, label="mean over subjects")
    ax.axvline(0, ls="--", color="0.4")
    ax.set(title=f"Max achievable r, {n}-trial test average", xlabel="ms after image onset", ylabel="r")
axes[0].legend()
fig.savefig(C.FIGURES_DIR / "01_noise_ceiling.png", dpi=150, bbox_inches="tight")

## Folds and label budgets

Ten hold-one-subject-out folds, and for each held-out subject and seed one
random order of its training images, so the k-subsets are nested (the first
k images of that order).

In [ ]:
import json

from cross_subject_encoding.data.splits import nested_subsets

subsets = nested_subsets(C.EEG_N_TRAIN_IMAGES, C.K_GRID, C.EEG_SUBJECTS, C.SEEDS)
subsets["folds"] = C.EEG_FOLDS
with open(C.DATA_PROCESSED / C.EEG_K_SUBSETS, "w") as f:
    json.dump(subsets, f)
first = subsets["orders"][C.EEG_SUBJECTS[0]][str(C.SEED)]
print(f"{len(subsets['folds'])} folds; stored {subsets['k_max_stored']} positions per subject and seed; "
      f"first images for {C.EEG_SUBJECTS[0]}: {first[:5]}")

## Save outputs as a Kaggle Dataset

Publishes every file this notebook wrote to `data/processed/`, `results/metrics/` and `results/figures/` as a new version of the private Kaggle Dataset `cross-subject-brain-encoding-01-outputs` (created on the first run). The cell first checks that the required outputs exist, so an unfinished run uploads nothing.

Needs Internet on and the Kaggle Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` attached to this notebook (Add-ons -> Secrets). Without them the cell prints how to set them up and skips.

In [ ]:
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

EXPORT_TO_DATASET = True
if EXPORT_TO_DATASET:
    export_to_dataset(
        slug="cross-subject-brain-encoding-01-outputs",
        title="cross-subject-brain-encoding 01 outputs",
        files=[
            C.DATA_PROCESSED / C.EEG_NOISE_CEILING,
            C.DATA_PROCESSED / C.EEG_K_SUBSETS,
            C.FIGURES_DIR / "01_noise_ceiling.png",
        ],
        include_dirs=[C.DATA_PROCESSED, C.METRICS_DIR, C.FIGURES_DIR],  # everything else this notebook wrote
        message="notebook run",
    )